# 街景字符识别 - FPN Multi-Head 模型（v8更新版）

基于 PyTorch 的街景字符识别项目，采用 FPN + Multi-Head Attention 架构。

## 重要更新说明（v8）
- ✅ **模型架构优化**：Head数量从6减少到3，匹配94%样本长度≤3的分布
- ✅ **OOM问题修复**：降低batch size，启用显存优化
- ✅ **训练稳定性**：降低学习率，放宽梯度裁剪
- ✅ **正则化增强**：提高dropout和weight decay
- ✅ **注意力监督**：禁用强制高斯分布拟合
- ✅ **数据增强**：大幅降低CutMix概率以避免BBox损失为0

**支持三种模型**：`fpn_multihead`（默认）、`transformer`、`ctc`

**支持两种GPU环境**：
- NVIDIA A100 (8核 / 32GB内存 / 24GB显存)
- AMD ROCm (8核 / 200GB内存 / 192GB显存)

运行时自动检测GPU环境并选择对应的参数配置。

**torch.compile 支持**：AMD ROCm 环境下自动启用 `torch.compile`，首次运行需编译 kernel（约5-15分钟），后续运行使用缓存。

**日志路径**：`{项目目录}/logs/train_YYYYMMDD_HHMMSS.log`

In [ ]:
# 环境配置：设置随机种子、检测GPU、打印版本信息
# ⚠️ 注意：必须在 import torch 前设置环境变量
import sys
import os

# 🚀 先设置显存优化环境变量（非常重要！）
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print(f'✅ 环境变量 PYTORCH_CUDA_ALLOC_CONF = {os.environ.get("PYTORCH_CUDA_ALLOC_CONF")}')

import torch as t
from config import config, BASE_DIR, data_dir, IS_MODELSCOPE, GPU_PLATFORM, TOTAL_VRAM_GB, IS_NVIDIA, IS_AMD, set_seed
from utils.platform import get_precision_config, is_nvidia_cuda

# 设置随机种子
set_seed(42)

# 平台特定精度配置
precision_config = get_precision_config()
if is_nvidia_cuda():
    try:
        t.set_float32_matmul_precision('high')
        print('✅ TF32 matmul precision enabled')
    except Exception:
        print('⚠️ TF32 matmul precision not supported on this GPU')
    try:
        t.backends.cudnn.allow_tf32 = True
        print('✅ CuDNN TF32 enabled')
    except Exception:
        print('⚠️ CuDNN TF32 not supported')
else:
    print('ℹ️ TF32/CuDNN: platform-specific settings skipped')

print('=' * 70)
print('📋 环境信息 (v8 OOM优化版)')
print('=' * 70)
print(f'Python: {sys.version}')
print(f'PyTorch: {t.__version__}')
print(f'GPU Platform: {GPU_PLATFORM.upper()}')
print(f'CUDA available: {t.cuda.is_available()}')
if t.cuda.is_available():
    print(f'CUDA version: {t.version.cuda}')
    print(f'GPU: {t.cuda.get_device_name(0)}')
    props = t.cuda.get_device_properties(0)
    vram = getattr(props, 'total_mem', getattr(props, 'total_memory', 0)) / (1024**3)
    print(f'GPU VRAM: {vram:.1f} GB (预留: {config.oom_headroom_ratio*100:.0f}%)')
print(f'BASE_DIR: {BASE_DIR}')
print(f'ModelScope环境: {IS_MODELSCOPE}')
print()
print('🎯 训练配置:')
print(f'  Model type: {config.model_type}')
print(f'  Batch size: {config.batch_size} (eval: {config.eval_batch_size})')
print(f'  Gradient accumulation: {config.grad_accum_steps} (effective: {config.batch_size * config.grad_accum_steps})')
print(f'  Num heads: {config.num_heads}')
print(f'  Input size: {config.input_height}x{config.input_width}')
print(f'  Use torch.compile: {config.use_torch_compile}')
print(f'  Use gradient checkpoint: {config.use_gradient_checkpoint}')
print(f'  Optimizer: {config.optimizer_type}')
print(f'  Scheduler: {config.scheduler_type}')
print(f'  Learning rate: {config.lr} (warmup: {config.warmup_epochs} epochs)')
print()
print('🔧 超参数:')
print(f'  Dropout: {config.dropout}')
print(f'  Weight decay: {config.weights_decay}')
print(f'  CutMix prob: {config.cutmix_prob}')
print(f'  Label smooth: {config.smooth}')
print(f'  Early stopping patience: {config.early_stopping_patience}')
print()
print('⚖️  损失权重:')
print(f'  cls_loss_weight: {config.cls_loss_weight}')
print(f'  bbox_loss_weight: {config.bbox_loss_weight}')
print(f'  aux_loss_weight: {config.aux_loss_weight}')
print(f'  ordering_loss_weight: {config.ordering_loss_weight}')
print(f'  attn_supervision_weight: {config.attn_supervision_weight}')
print(f'  attn_diversity_weight: {config.attn_diversity_weight}')
print('=' * 70)

In [ ]:
# 下载数据集（仅在数据不存在时执行）
from data.download import download_dataset
download_dataset()

## 数据探索

In [ ]:
# 统计训练集、验证集、测试集的图片数量
from glob import glob

train_list = glob(os.path.join(data_dir['train_data'], '*.png'))
val_list = glob(os.path.join(data_dir['val_data'], '*.png'))
test_list = glob(os.path.join(data_dir['test_data'], '*.png'))

print(f'训练集图片数: {len(train_list)}')
print(f'验证集图片数: {len(val_list)}')
print(f'测试集图片数: {len(test_list)}')

In [ ]:
# 分析训练集中数字位数的分布
import json

with open(data_dir['train_label'], 'r') as f:
    marks = json.load(f)

dicts = {}
for img, mark in marks.items():
    n = len(mark['label'])
    dicts[n] = dicts.get(n, 0) + 1
for k, v in sorted(dicts.items()):
    print(f'{k}位数字的图片数目: {v}')
print()
print(f'长度≤3的样本数: {sum(v for k, v in dicts.items() if k<=3)} (占比: {sum(v for k, v in dicts.items() if k<=3)/sum(dicts.values())*100:.1f}%)')

## 模型定义

创建模型并统计参数量

In [ ]:
# 创建模型并统计参数量
from models import create_model

model_type = config.model_type  # 从config读取，避免硬编码
model = create_model(model_type)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'模型类型: {model_type}')
print(f'总参数量: {total_params:,}')
print(f'可训练参数量: {trainable_params:,}')
print(f'模型大小: {total_params * 4 / (1024**2):.1f} MB (FP32), {total_params * 2 / (1024**2):.1f} MB (FP16)')

## 训练 FPN Multi-Head 模型

### 训练模式选择
- **从头训练**（推荐）：禁用checkpoint加载，使用优化后的超参数
- **续训**：从之前的checkpoint继续（仅在中断恢复时使用）

### 训练日志
训练过程中日志会自动记录到 `logs/` 目录下，包含：
- 每个batch的loss、学习率、精度、GPU/CPU内存使用
- 每个epoch的训练/验证精度（joint acc + char acc）、耗时、early stopping状态
- 显存使用情况（OOM优化版本新增）

In [ ]:
# 训练FPN Multi-Head模型
# 日志自动保存到 {项目目录}/logs/ 目录
#
# resume_weights_only=False: 完整恢复optimizer/scheduler/scaler状态继续训练
# resume_weights_only=True: 只加载模型权重，用新optimizer从头训练

from trainer.multihead import MultiHeadTrainer
from utils.misc import find_latest_checkpoint

# 🎯 设置：是否从头训练
TRAIN_FROM_SCRATCH = True  # 推荐设置为True

if TRAIN_FROM_SCRATCH:
    config.pretrained = None
    config.start_epoch = 0
    config.resume_weights_only = False
    print('✅ 从头训练模式：不加载checkpoint')
else:
    # 续训模式
    latest = find_latest_checkpoint(config.checkpoints)
    if latest:
        config.pretrained = latest
        config.resume_weights_only = False
        print(f'📂 从最新 checkpoint 续训: {latest}')
    else:
        config.pretrained = None
        config.start_epoch = 0
        config.resume_weights_only = False
        print('ℹ️ 无 checkpoint，自动切换到从头训练模式')

trainer = MultiHeadTrainer(model_type=model_type)
trainer.train()

## 训练诊断（v8增强）

验证数据加载、LR调度、模型参数更新等关键功能。

In [ ]:
# 诊断：验证DataLoader每epoch数据顺序不同
import torch as t
from utils.seed import make_epoch_generator

print('=== DataLoader Shuffle 诊断 ===')
print(f'训练集大小: {len(trainer.train_set)}')
print(f'Batch size: {config.batch_size}')

first_indices = []
for epoch in range(3):
    gen = make_epoch_generator(42, epoch=epoch)
    sampler = t.utils.data.RandomSampler(trainer.train_set, generator=gen)
    indices = list(sampler)[:config.batch_size]
    first_indices.append(indices)
    print(f'Epoch {epoch}: 前{config.batch_size}个样本索引 = {indices[:5]}...')

if first_indices[0] == first_indices[1]:
    print('⚠️ 警告: epoch 0 和 epoch 1 的数据顺序相同！')
else:
    print('✅ 每epoch数据顺序不同，shuffle正常')

In [ ]:
# 诊断：验证LR调度是否正确
print('=== LR Scheduler 诊断 ===')
print(f'Scheduler type: {config.scheduler_type}')
print(f'Base LR: {config.lr}')
print(f'Backbone LR factor: {config.backbone_lr_factor}')
print(f'Warmup epochs: {config.warmup_epochs}')
print(f'Scheduler last_epoch: {trainer.lr_scheduler.last_epoch}')
print(f'Current backbone LR: {trainer.optimizer.param_groups[0]["lr"]:.8f}')
print(f'Current head LR: {trainer.optimizer.param_groups[1]["lr"]:.8f}')
print()
print('LR schedule (first 10 epochs):')
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, CosineAnnealingWarmRestarts, SequentialLR
test_opt = t.optim.SGD([{'params': [t.randn(1)], 'lr': config.lr * config.backbone_lr_factor},
                         {'params': [t.randn(1)], 'lr': config.lr}], momentum=0.9)
if config.scheduler_type == 'warm_restarts':
    warmup = LinearLR(test_opt, start_factor=config.warmup_start_factor, total_iters=config.warmup_epochs)
    restart = CosineAnnealingWarmRestarts(test_opt, T_0=config.scheduler_T0, T_mult=config.scheduler_T_mult, eta_min=config.scheduler_eta_min)
    test_sched = SequentialLR(test_opt, schedulers=[warmup, restart], milestones=[config.warmup_epochs])
else:
    warmup = LinearLR(test_opt, start_factor=config.warmup_start_factor, total_iters=config.warmup_epochs)
    cosine = CosineAnnealingLR(test_opt, T_max=config.epoches - config.warmup_epochs, eta_min=config.scheduler_eta_min)
    test_sched = SequentialLR(test_opt, schedulers=[warmup, cosine], milestones=[config.warmup_epochs])
for e in range(min(10, config.epoches)):
    if e > 0:
        test_sched.step()
    print(f'  Epoch {e+1}: backbone_lr={test_opt.param_groups[0]["lr"]:.8f}, head_lr={test_opt.param_groups[1]["lr"]:.8f}')

In [ ]:
# 诊断：验证模型参数在训练中是否更新
raw = trainer._get_raw_model()
first_param = next(iter(raw.parameters()))
print(f'模型首个参数 shape: {first_param.shape}')
print(f'首个参数前5个值: {first_param.data.flatten()[:5].tolist()}')
print(f'参数范数: {first_param.data.norm().item():.4f}')
print()
total_norm = sum(p.data.norm().item() ** 2 for p in raw.parameters()) ** 0.5
print(f'模型总参数范数: {total_norm:.2f}')
print()
if trainer.train_log:
    print('最近5条训练日志:')
    for entry in trainer.train_log[-5:]:
        print(f'  Epoch {entry["epoch"]}: train_joint={entry["train_joint_acc"]:.2f}, '
              f'val_joint={entry["val_joint_acc"]:.2f}, lr={entry["lr"]:.6f}')

## 评估

In [ ]:
# 在验证集上评估模型精度（返回joint acc）
val_acc = trainer._eval()
print(f'最佳验证精度: {trainer.best_acc * 100:.2f}%')

In [ ]:
# 逐Head评估每个数字位的分类精度
trainer.eval_detailed()

## 推理与提交

In [ ]:
# 推理（use_compile=True 时启用 torch.compile 加速推理）
from inference.predict import predicts, ctc_predict, ensemble_predict

# model_path = 'checkpoints/best-resnet101-acc-xx.xx.pth'
# results = predicts(model_path, 'submit.csv', use_tta=True, use_compile=True)

## 训练日志

In [ ]:
# 查看训练日志
if trainer.train_log:
    for entry in trainer.train_log[-5:]:
        print(entry)
else:
    print('暂无训练日志')